In [ ]:
import flax.linen as nn
import optax
import h5py as hf
import matplotlib.pyplot as plt
import znnl as nl
import numpy as np
from znnl.data.decision_boundary import (
    DecisionBoundaryGenerator,
    circle,
    linear_boundary,
)

# Dask support
from dask.distributed import Client
from dask_jobqueue import SLURMCluster

In [ ]:
ds_size=500

In [ ]:
def train(index: int):
    """
    Run the experiment.
    """

    class Network(nn.Module):
        """
        Perceptron network.
        """

        @nn.compact
        def __call__(self, x):
            """
            Call method for the network
            """
            x = nn.Dense(100, use_bias=False)(x)
            x = nn.relu(x)
            x = nn.Dense(2, use_bias=False)(x)
            return x
        
    generator = DecisionBoundaryGenerator(ds_size, discriminator="line",  one_hot=True)
    model = nl.models.FlaxModel(
        flax_module=Network(), 
        optimizer=optax.adam(0.01), 
        input_shape=(1, 2)
    )
    # Prepare the recorders
    train_recorder = nl.training_recording.JaxRecorder(
        name=f"ce-single-layer-100/train_recorder_{index}",
        loss=True,
        entropy=True,
        trace=True,
        accuracy=True,
        magnitude_variance=True,
        update_rate=1,
    )
    test_recorder = nl.training_recording.JaxRecorder(
        name=f"ce-single-layer-100/test_recorder_{index}", loss=True, accuracy=True, update_rate=1,
    )
    train_recorder.instantiate_recorder(data_set=generator.train_ds)
    test_recorder.instantiate_recorder(data_set=generator.test_ds)

    trainer = nl.training_strategies.SimpleTraining(
        model=model,
        loss_fn=nl.loss_functions.CrossEntropyLoss(),
        accuracy_fn=nl.accuracy_functions.LabelAccuracy(),
        recorders=[train_recorder, test_recorder],
    )

    _ = trainer.train_model(
        train_ds=generator.train_ds,
        test_ds=generator.test_ds,
        batch_size=128,
        epochs=5000,
    )


## Deployment

In [ ]:
def deploy_jobs(indices):
    cluster = SLURMCluster(
        cores=16,
        processes=1,
        memory="64GB",
        queue="single",
        walltime="03:00:00",
        death_timeout="15s",
        worker_extra_args=["--resources GPU=1"],
        log_directory=f'./ce-single-layer-100/dask-logs',
        job_script_prologue=["module load devel/cuda/12.1"],
        job_extra_directives=["--gres=gpu:1"]
    )
    cluster.scale(1)
    client = Client(cluster)

    results =  [
        client.submit(
            train, index,  resources={"GPU": 1}
            ) for index in indices
        ]

    return results


In [ ]:
results = []


indice_blocks = np.split(np.linspace(1, 20, 20, dtype=int), 10)

for block in indice_blocks:
    results.append(deploy_jobs(block))

In [ ]:
indice_blocks

In [ ]:
results